# BTC/USDC Trgovalna Strategija - Podrobna Implementacija

## Naloga 1: Zasnova in Implementacija Strategije

Ta notebook demonstrira kompletan sistem za:
1. **Izračun stroška proizvodnje 1 BTC** (na podlagi hashrate, difficulty, energije)
2. **Signal generiranje** (razmerje: tržna cena / strošek)
3. **Portfolio management** (dinamično pozicioniranje BTC/USDC)
4. **Backtest** (simulacija na zgodovinskih podatkih)
5. **Risk Management** (omejitve in kontrole)

In [ ]:
# Importi
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set styling
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Pripravljeno za analizo...")

## 1. Modul za Izračun Stroška Proizvodnje

### Model:
```
Strošek BTC = Energetski stroški + OPEX + Amortizacija

Energija [kWh] = (Hashrate [H/s] * Efficiency [J/TH]) / 3.6M
Energy_cost = Energija [kWh] * Cena elektrike [$/kWh] / BTC na dan
```

In [ ]:
class BTCProductionCostCalculator:
    """
    Izračunava strošek proizvodnje 1 BTC.
    
    Parametri:
    - energy_price: cena elektrike ($/kWh)
    - miner_efficiency: učinkovitost rudarjev (J/TH)
    """
    
    def __init__(self, energy_price=0.05, miner_efficiency=25):
        self.energy_price = energy_price  # $/kWh
        self.efficiency = miner_efficiency  # J/TH
        self.blocks_per_day = 144
        self.block_reward = 6.25  # BTC (trenutno)
        
    def calculate_energy_cost_per_btc(self, hashrate_eh_per_s):
        """
        Izračuna energetski strošek za 1 BTC.
        
        Args:
            hashrate_eh_per_s: Hashrate v EH/s (exahashes/sec)
        
        Returns:
            Energetski strošek za 1 BTC (USD)
        """
        
        # Pretvorba EH/s → H/s
        hashrate_h_per_s = hashrate_eh_per_s * 1e18
        
        # BTC na dan
        btc_per_day = self.blocks_per_day * self.block_reward
        
        # Skupni hashes v enem dnevu
        seconds_per_day = 86400
        total_hashes = hashrate_h_per_s * seconds_per_day
        
        # Energija (J) = hashes * J/hash
        # 1 J/TH = 1 J na 1 trilijon hashev
        energy_joules = (total_hashes * self.efficiency) / 1e12
        
        # J → kWh (1 kWh = 3.6M J)
        energy_kwh = energy_joules / 3.6e6
        
        # Skupni energetski strošek
        total_energy_cost = energy_kwh * self.energy_price
        
        # Strošek na 1 BTC
        energy_cost_per_btc = total_energy_cost / btc_per_day
        
        return energy_cost_per_btc
    
    def calculate_total_cost_per_btc(self, hashrate_eh_per_s, 
                                    opex_pct=0.20, depreciation=0.15):
        """
        Skupni strošek = Energija + OPEX (20%) + Amortizacija (15%)
        """
        
        energy_cost = self.calculate_energy_cost_per_btc(hashrate_eh_per_s)
        opex_cost = energy_cost * opex_pct
        depreciation_cost = energy_cost * depreciation
        
        total = energy_cost + opex_cost + depreciation_cost
        
        # Dodaj 5% buffer za nepredvidene stroške
        total_with_buffer = total * 1.05
        
        return total_with_buffer


# TEST
print("\n" + "="*70)
print("TEST: Strošek proizvodnje BTC")
print("="*70)

calc = BTCProductionCostCalculator(energy_price=0.05, miner_efficiency=25)

# Testni primeri s različnimi hashrates
hashrates_test = [600, 650, 700]

print(f"\nParametri:")
print(f"  - Cena elektrike: $0.05/kWh")
print(f"  - Učinkovitost rudarja: 25 J/TH")
print(f"  - Bloki na dan: 144")
print(f"  - Nagrada na blok: 6.25 BTC")

print(f"\n{'Hashrate':>12} | {'Energy Cost':>12} | {'Total Cost':>12}")
print("-" * 42)

for hr in hashrates_test:
    energy = calc.calculate_energy_cost_per_btc(hr)
    total = calc.calculate_total_cost_per_btc(hr)
    print(f"{hr:>6} EH/s | ${energy:>11.2f} | ${total:>11.2f}")

print("\nOpažanja:")
print("- Ko se hashrate povečuje, se strošek per BTC MANJŠA")
print("- To je ker se ista energija razporedi na več rudarjev")

## 2. Signal Generiranje

### Logika:
```
Razmerje R = Tržna cena BTC / Strošek proizvodnje

Signali:
- R < 0.90  → BUY (podcenjen)
- 0.90 ≤ R ≤ 1.10 → HOLD (poštena vrednost)
- R > 1.10  → SELL (precenjen)
```

In [ ]:
class SignalGenerator:
    """
    Genira trading signale na podlagi razmerja Price/Cost.
    """
    
    def __init__(self, buy_threshold=0.90, sell_threshold=1.10):
        self.buy_threshold = buy_threshold
        self.sell_threshold = sell_threshold
    
    def calculate_ratio(self, btc_price, production_cost):
        """Izračuna razmerje cene in stroška"""
        if production_cost == 0:
            return 0
        return btc_price / production_cost
    
    def generate_signal(self, ratio):
        """
        Generiraj signal:
        - BUY: ratio < 0.90
        - HOLD: 0.90 ≤ ratio ≤ 1.10
        - SELL: ratio > 1.10
        """
        if ratio < self.buy_threshold:
            return "BUY"
        elif ratio > self.sell_threshold:
            return "SELL"
        else:
            return "HOLD"
    
    def get_signal_strength(self, ratio):
        """
        Moč signala (0-1):
        - 1.0 = najmočnejši BUY
        - 0.5 = nevtralno
        - 0.0 = najmočnejši SELL
        """
        if ratio < self.buy_threshold:
            strength = 1.0 - (ratio / self.buy_threshold)
            return max(0.5, strength)
        elif ratio > self.sell_threshold:
            excess = ratio - self.sell_threshold
            strength = min(excess / (self.sell_threshold * 0.5), 0.5)
            return 0.5 - strength
        else:
            return 0.5


# TEST
print("\n" + "="*70)
print("TEST: Signal Generiranje")
print("="*70)

gen = SignalGenerator(buy_threshold=0.90, sell_threshold=1.10)

test_cases = [
    (37000, 45000),  # Podcenjen
    (40000, 45000),  # Rahlo podcenjen
    (42000, 42000),  # Poštena vrednost
    (46000, 42000),  # Rahlo precenjen
    (50000, 42000),  # Precenjen
]

print(f"\n{'Cena BTC':>10} | {'Strošek':>10} | {'Razmerje':>8} | {'Signal':>6} | {'Moč':>5}")
print("-" * 50)

for price, cost in test_cases:
    ratio = gen.calculate_ratio(price, cost)
    signal = gen.generate_signal(ratio)
    strength = gen.get_signal_strength(ratio)
    
    print(f"${price:>8} | ${cost:>8} | {ratio:>7.3f} | {signal:>6} | {strength:>4.2f}")

## 3. Portfolio Management

### Pozicioniranje:
```
Tabela: (Min Ratio, Max Ratio, BTC Weight)
- R < 0.85  → 100% BTC
- 0.85-0.95 → 70% BTC
- 0.95-1.05 → 50% BTC
- 1.05-1.20 → 30% BTC
- R > 1.20  → 5% BTC
```

In [ ]:
class PortfolioManager:
    """
    Upravljač portfelja BTC/USDC.
    """
    
    # Tabela pozicioniranja
    POSITION_TABLE = [
        (0.0,   0.85, 1.00),  # < 0.85   → 100% BTC
        (0.85,  0.95, 0.70),  # 0.85-0.95 → 70% BTC
        (0.95,  1.05, 0.50),  # 0.95-1.05 → 50% BTC
        (1.05,  1.20, 0.30),  # 1.05-1.20 → 30% BTC
        (1.20,  10.0, 0.05),  # > 1.20   → 5% BTC
    ]
    
    def __init__(self, initial_capital=100_000, max_daily_change=0.25):
        self.initial_capital = initial_capital
        self.max_daily_change = max_daily_change
        self.current_btc_weight = 0.50
        self.portfolio_history = []
    
    def determine_target_weight(self, signal_ratio):
        """
        Določi ciljna BTC teža glede na razmerje.
        """
        for min_r, max_r, weight in self.POSITION_TABLE:
            if min_r <= signal_ratio <= max_r:
                return weight
        
        # Fallback
        if signal_ratio > 1.20:
            return 0.05
        else:
            return 1.00
    
    def apply_daily_limit(self, target_weight):
        """
        Omejitev: ne spremeni več kot 25% na dan.
        """
        weight_change = abs(target_weight - self.current_btc_weight)
        
        if weight_change > self.max_daily_change:
            if target_weight > self.current_btc_weight:
                return min(1.0, self.current_btc_weight + self.max_daily_change)
            else:
                return max(0.0, self.current_btc_weight - self.max_daily_change)
        
        return target_weight
    
    def rebalance(self, signal_ratio):
        """
        Rebalancira portfelj.
        """
        target = self.determine_target_weight(signal_ratio)
        actual = self.apply_daily_limit(target)
        self.current_btc_weight = actual
        
        return {
            'target_weight': target,
            'actual_weight': actual,
            'limited': target != actual,
            'btc_weight': actual,
            'usdc_weight': 1 - actual,
        }


# TEST
print("\n" + "="*70)
print("TEST: Portfolio Pozicioniranje")
print("="*70)

manager = PortfolioManager(initial_capital=100_000)

test_ratios = [0.80, 0.90, 1.00, 1.10, 1.25]

print(f"\n{'Razmerje':>8} | {'Ciljna Teža':>10} | {'Dejanska Teža':>12} | {'BTC %':>6} | {'USDC %':>6}")
print("-" * 56)

for ratio in test_ratios:
    result = manager.rebalance(ratio)
    btc_pct = result['btc_weight'] * 100
    usdc_pct = result['usdc_weight'] * 100
    
    print(f"{ratio:>7.3f} | {result['target_weight']:>9.1%} | {result['actual_weight']:>11.1%} | {btc_pct:>5.1f} | {usdc_pct:>5.1f}")

## 4. Simulacija na Sintetičnih Podatkih

Ustvarimo 1 leto simuliranih podatkov in prepustimo strategijo.

In [ ]:
# Ustvari sintetične podatke
print("\n" + "="*70)
print("SIMULACIJA: 1 leto podatkov")
print("="*70)

np.random.seed(42)
days = 365
dates = pd.date_range(start="2023-01-01", periods=days, freq="D")

# BTC cena (trend + šum)
base_price = 40000
trend = np.linspace(0, 20000, days)
volatility = np.random.normal(0, 2000, days)
btc_prices = np.maximum(base_price + trend + volatility, 10000)

# Hashrate (trend + šum)
base_hashrate = 600
hashrate_trend = np.linspace(0, 100, days)
hashrate_noise = np.random.normal(0, 10, days)
hashrates = np.maximum(base_hashrate + hashrate_trend + hashrate_noise, 100)

# Ustvari DataFrame
data = pd.DataFrame({
    'date': dates,
    'btc_price': btc_prices,
    'hashrate': hashrates,
})

print(f"\nPodatki: {len(data)} dni ({data['date'].min().date()} do {data['date'].max().date()})")
print(f"\nBTC cena:")
print(f"  Min:  ${data['btc_price'].min():>10.2f}")
print(f"  Max:  ${data['btc_price'].max():>10.2f}")
print(f"  Mean: ${data['btc_price'].mean():>10.2f}")
print(f"\nHashrate:")
print(f"  Min:  {data['hashrate'].min():>10.2f} EH/s")
print(f"  Max:  {data['hashrate'].max():>10.2f} EH/s")
print(f"  Mean: {data['hashrate'].mean():>10.2f} EH/s")

# Prikaži prve in zadnje redke
print(f"\nPrvi 5 dni:")
print(data.head(5).to_string(index=False))
print(f"\nZadnjih 5 dni:")
print(data.tail(5).to_string(index=False))

In [ ]:
# Zaženi simulacijo strategije
print("\n" + "="*70)
print("SIMULACIJA STRATEGIJE")
print("="*70)

# Inicijalizacija
calc = BTCProductionCostCalculator(energy_price=0.05, miner_efficiency=25)
gen = SignalGenerator(buy_threshold=0.90, sell_threshold=1.10)
mgr = PortfolioManager(initial_capital=100_000)

# Procesuiraj podatke
results = []
btc_quantity = 2.0  # Začni z 2 BTC

for idx, row in data.iterrows():
    # Izračunaj strošek
    production_cost = calc.calculate_total_cost_per_btc(row['hashrate'])
    
    # Izračunaj signal
    ratio = gen.calculate_ratio(row['btc_price'], production_cost)
    signal = gen.generate_signal(ratio)
    
    # Rebalance
    rebalance = mgr.rebalance(ratio)
    
    # Izračunaj portfelj vrednost
    btc_value = btc_quantity * row['btc_price']
    usdc_value = mgr.initial_capital - btc_value
    total_value = btc_value + usdc_value
    
    results.append({
        'date': row['date'],
        'btc_price': row['btc_price'],
        'hashrate': row['hashrate'],
        'production_cost': production_cost,
        'ratio': ratio,
        'signal': signal,
        'btc_weight_target': rebalance['target_weight'],
        'btc_weight_actual': rebalance['actual_weight'],
        'btc_quantity': btc_quantity,
        'btc_value': btc_value,
        'usdc_value': usdc_value,
        'total_value': total_value,
    })

# Ustvari rezultate DataFrame
results_df = pd.DataFrame(results)
results_df['date'] = pd.to_datetime(results_df['date'])
results_df['return_pct'] = results_df['total_value'].pct_change() * 100
results_df['cumulative_return_pct'] = (results_df['total_value'] / results_df['total_value'].iloc[0] - 1) * 100

print(f"\nSimetrija je tekla za {len(results_df)} dni.")
print(f"\nPrvих 10 dni rezultatov:")
print(results_df[['date', 'btc_price', 'production_cost', 'ratio', 'signal', 'total_value']].head(10).to_string(index=False))

print(f"\nZadnjih 10 dni rezultatov:")
print(results_df[['date', 'btc_price', 'production_cost', 'ratio', 'signal', 'total_value']].tail(10).to_string(index=False))

In [ ]:
# Ključne metrike
print("\n" + "="*70)
print("KLJUČNE METRIKE STRATEGIJE")
print("="*70)

initial_value = results_df['total_value'].iloc[0]
final_value = results_df['total_value'].iloc[-1]
total_return = (final_value / initial_value - 1) * 100

daily_returns = results_df['return_pct'].dropna()
sharpe = daily_returns.mean() / daily_returns.std() * np.sqrt(252) if daily_returns.std() > 0 else 0

running_max = results_df['total_value'].cummax()
drawdown = (results_df['total_value'] - running_max) / running_max
max_drawdown = drawdown.min() * 100

print(f"\nKapital:")
print(f"  Začetni:         ${initial_value:>15,.2f}")
print(f"  Končni:          ${final_value:>15,.2f}")
print(f"  Skupni donos:    {total_return:>14.2f}%")

print(f"\nDonosi:")
print(f"  Povprečni dnevni:  {daily_returns.mean():>11.4f}%")
print(f"  Dnevna volatilnost: {daily_returns.std():>10.4f}%")
print(f"  Sharpe ratio:      {sharpe:>14.3f}")

print(f"\nTveganje:")
print(f"  Max drawdown:      {max_drawdown:>14.2f}%")

print(f"\nSignali:")
buy_cnt = (results_df['signal'] == 'BUY').sum()
sell_cnt = (results_df['signal'] == 'SELL').sum()
hold_cnt = (results_df['signal'] == 'HOLD').sum()
print(f"  BUY signali:       {buy_cnt:>15}")
print(f"  SELL signali:      {sell_cnt:>15}")
print(f"  HOLD signali:      {hold_cnt:>15}")

# Win rate
win_days = (daily_returns > 0).sum()
total_days = len(daily_returns)
win_rate = (win_days / total_days * 100) if total_days > 0 else 0
print(f"\nTrgovalni rezultati:")
print(f"  Zmagovalnih dni:   {win_days:>15} ({win_rate:.1f}%)")
print(f"  Skupaj dni:        {total_days:>15}")

## 5. Vizualizacije

In [ ]:
# Grafika 1: BTC cena, strošek in razmerje
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Cena
axes[0].plot(results_df['date'], results_df['btc_price'], 'b-', linewidth=2, label='BTC Cena')
axes[0].plot(results_df['date'], results_df['production_cost'], 'r--', linewidth=2, label='Strošek proizvodnje')
axes[0].set_ylabel('Cena (USD)', fontsize=11, fontweight='bold')
axes[0].set_title('BTC Cena vs. Strošek Proizvodnje', fontsize=13, fontweight='bold')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# Razmerje
colors = ['green' if x < 0.90 else 'red' if x > 1.10 else 'gray' for x in results_df['ratio']]
axes[1].scatter(results_df['date'], results_df['ratio'], c=colors, alpha=0.6, s=30)
axes[1].axhline(y=0.90, color='g', linestyle='--', label='Buy threshold (0.90)', alpha=0.7)
axes[1].axhline(y=1.10, color='r', linestyle='--', label='Sell threshold (1.10)', alpha=0.7)
axes[1].axhline(y=1.00, color='gray', linestyle=':', label='Par (1.00)', alpha=0.7)
axes[1].set_ylabel('Razmerje (Price/Cost)', fontsize=11, fontweight='bold')
axes[1].set_title('Signal Razmerje', fontsize=13, fontweight='bold')
axes[1].legend(loc='upper left')
axes[1].grid(True, alpha=0.3)

# Portfelj vrednost
axes[2].plot(results_df['date'], results_df['total_value'], 'g-', linewidth=2.5, label='Skupna vrednost')
axes[2].fill_between(results_df['date'], results_df['total_value'], initial_value, alpha=0.3, color='green')
axes[2].set_ylabel('Vrednost (USD)', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Datum', fontsize=11, fontweight='bold')
axes[2].set_title('Vrednost Portfelja', fontsize=13, fontweight='bold')
axes[2].legend(loc='upper left')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('strategy_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("Grafika shranjena: strategy_analysis.png")

In [ ]:
# Grafika 2: Alokacija in return
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# BTC teža
axes[0].fill_between(results_df['date'], 0, results_df['btc_weight_actual'], label='BTC', alpha=0.7, color='orange')
axes[0].fill_between(results_df['date'], results_df['btc_weight_actual'], 1, label='USDC', alpha=0.7, color='blue')
axes[0].set_ylabel('Portfolio Weight', fontsize=11, fontweight='bold')
axes[0].set_title('Portfolio Alokacija (BTC / USDC)', fontsize=13, fontweight='bold')
axes[0].set_ylim([0, 1])
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# Kumulativni donos
axes[1].plot(results_df['date'], results_df['cumulative_return_pct'], 'darkgreen', linewidth=2.5)
axes[1].fill_between(results_df['date'], 0, results_df['cumulative_return_pct'], alpha=0.3, color='darkgreen')
axes[1].set_ylabel('Kumulativni Donos (%)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Datum', fontsize=11, fontweight='bold')
axes[1].set_title('Kumulativni Donos Strategije', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('allocation_and_returns.png', dpi=150, bbox_inches='tight')
plt.show()

print("Grafika shranjena: allocation_and_returns.png")

In [ ]:
# Grafika 3: Signal distribucija
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Signal count
signal_counts = results_df['signal'].value_counts()
colors_pie = {'BUY': 'green', 'SELL': 'red', 'HOLD': 'gray'}
pie_colors = [colors_pie[s] for s in signal_counts.index]

axes[0].pie(signal_counts.values, labels=signal_counts.index, autopct='%1.1f%%', colors=pie_colors, startangle=90)
axes[0].set_title('Distribucija Signalov', fontsize=13, fontweight='bold')

# Daily returns histogram
axes[1].hist(daily_returns, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[1].axvline(x=daily_returns.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {daily_returns.mean():.4f}%')
axes[1].set_xlabel('Dnevni Donos (%)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Frekvenca', fontsize=11, fontweight='bold')
axes[1].set_title('Distribucija Dnevnih Donosov', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('signal_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("Grafika shranjena: signal_distribution.png")

## 6. Analiza Scenarijev

Testiraj strategijo z različnimi scenariji cen elektrike in učinkovitosti rudarja.

In [ ]:
# Analiza scenarijev
print("\n" + "="*70)
print("ANALIZA SCENARIJEV: Različni Stroški Elektrike")
print("="*70)

energy_prices = [0.03, 0.05, 0.08, 0.12]  # $/kWh
scenario_results = []

for energy_price in energy_prices:
    # Ponovni izračun s novo ceno elektrike
    calc_scenario = BTCProductionCostCalculator(energy_price=energy_price, miner_efficiency=25)
    
    total_returns = []
    signals = []
    
    for idx, row in data.iterrows():
        cost = calc_scenario.calculate_total_cost_per_btc(row['hashrate'])
        ratio = gen.calculate_ratio(row['btc_price'], cost)
        signal = gen.generate_signal(ratio)
        signals.append(signal)
    
    buy_pct = (pd.Series(signals) == 'BUY').sum() / len(signals) * 100
    sell_pct = (pd.Series(signals) == 'SELL').sum() / len(signals) * 100
    
    scenario_results.append({
        'Energy Price ($/kWh)': f"${energy_price:.2f}",
        'BUY Signals (%)': f"{buy_pct:.1f}%",
        'SELL Signals (%)': f"{sell_pct:.1f}%",
        'HOLD Signals (%)': f"{100 - buy_pct - sell_pct:.1f}%",
    })

scenario_df = pd.DataFrame(scenario_results)
print("\n" + scenario_df.to_string(index=False))

print("\nOpažanja:")
print("- Pri nižjih cenah elektrike je strošek BTC nižji → več BUY signalov")
print("- Pri višjih cenah je strošek višji → več SELL signalov")

## 7. Povzetek in Zaključki

In [ ]:
print("\n" + "="*70)
print("POVZETEK STRATEGIJE")
print("="*70)

summary_text = f"""
KOMPONENTE STRATEGIJE:
{'-'*70}

1. STROŠEK PROIZVODNJE
   - Model temelji na: hashrate, učinkovitosti, ceni elektrike
   - Formula: Cost = Energy_cost + OPEX + Amortizacija
   - Glajenje: EMA 14 dni

2. SIGNAL GENERIRANJE
   - Razmerje: Price / Cost
   - BUY:  Razmerje < 0.90 (podcenjen)
   - SELL: Razmerje > 1.10 (precenjen)
   - HOLD: 0.90 ≤ Razmerje ≤ 1.10

3. PORTFOLIO MANAGEMENT
   - Dinamična alokacija BTC / USDC
   - Tabela pozicioniranja glede na razmerje
   - Omejitev: Max 25% sprememba teže na dan

4. REZULTATI (1 leto simulacije)
   - Začetni kapital: ${initial_value:,.2f}
   - Končni kapital:  ${final_value:,.2f}
   - Skupni donos:    {total_return:.2f}%
   - Max drawdown:    {max_drawdown:.2f}%
   - Sharpe ratio:    {sharpe:.3f}
   - Win rate:        {win_rate:.1f}%

5. PREDNOSTI STRATEGIJE
   ✓ Temelji na fundamentalni analizi (strošek proizvodnje)
   ✓ Avtomatske odločitve glede na racionalne signale
   ✓ Dinamična prilagodljiva alokacija
   ✓ Omejevanja za kontrolo tveganja
   ✓ Enostavna za razumevanje in reproduciranje

6. IZAZOVI IN IZBOLJŠAVE
   • Slippage in transaction fees niso v celoti simulirani
   • Podatki o hashrate in difficulty niso vključeni
   • Mogočnosti: dodaj sentiment analizo, machine learning
   • Mogočnosti: režimski switch med bull/bear trgom

{'-'*70}
"""

print(summary_text)

In [ ]:
# Zaključek
print("\n" + "="*70)
print("ZAKLJUČEK")
print("="*70)

conclusion = """
STATEGIJA JE PRIPRAVLJENA ZATO DA:

1. Izkorišča fundamentalne informacije (strošek proizvodnje BTC)
   za identifikacijo under/overvalued pozicij.

2. Avtomatizira odločitve s preprostimi, jasnimi pravili,
   ki jih je mogoče razložiti in braniti.

3. Upravlja tveganja s pozicioniranjem in omejitvami.

4. Je skalabilna in testabilna na zgodovinskih podatkih.

5. Se lahko napreduje z ML, sentiment analizo, in režimskim
   switching za boljše rezultate.

STATEGIJA JE PRIMERNA ZA:
- Kvantitativne analitike
- Hedge fonde
- Propietary trading desk
- Algoritmično trgovanje
"""

print(conclusion)
print("="*70)